# AE2 - Exploracion descriptiva del dataset de Churn

**Asignatura:** Proyecto Integrado III - Analitica de Datos (PREICA2602B020038)

Primera etapa del proyecto integrador. Dataset: `scikit-learn/churn-prediction` (Telco Customer Churn). Flujo: carga, exploracion descriptiva, transformaciones de preparacion y generacion del resumen con `DatasetSummarizer`.

## 1. Importar las clases desde `../src`

In [1]:
import sys
import pathlib

SRC = (pathlib.Path.cwd().parent / 'src').resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# --- Alternativa para Google Colab (descomentar si se ejecuta en Colab) ---
# !git clone https://github.com/Jeremy-Pedraza/PREICA2602B020038-Analisis-de-Datos.git
# SRC = pathlib.Path('/content/PREICA2602B020038-Analisis-de-Datos/AE2/src').resolve()
# sys.path.insert(0, str(SRC))

from hf_dataset_loader import HFDatasetLoader
from dataset_summarizer import DatasetSummarizer

print('Clases importadas desde:', SRC)

Clases importadas desde: G:\Mi unidad\iUDigital Software\2026 S2\ProgramacionAnalisisdeDatos\EV1\AE2\src


## 2. Cargar el dataset y mostrar metadatos

In [2]:
loader = HFDatasetLoader(dataset_id='scikit-learn/churn-prediction', split='train')
df_crudo = loader.to_dataframe()
print('Metadatos:', loader.get_metadata())
df_crudo.head(10)

Metadatos: {'dataset_id': 'scikit-learn/churn-prediction', 'split': 'train', 'config': None, 'revision': None, 'fecha_consulta': '2026-10-24T16:02:41.145849+00:00', 'n_filas': 7043, 'n_columnas': 21, 'es_muestra': False}


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
5,9305-CDSKC,Female,0,No,No,8,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes
6,1452-KIOVK,Male,0,No,Yes,22,Yes,Yes,Fiber optic,No,...,No,No,Yes,No,Month-to-month,Yes,Credit card (automatic),89.10,1949.4,No
7,6713-OKOMC,Female,0,No,No,10,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,No,Mailed check,29.75,301.9,No
8,7892-POOKP,Female,0,Yes,No,28,Yes,Yes,Fiber optic,No,...,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes
9,6388-TABGU,Male,0,No,Yes,62,Yes,No,DSL,Yes,...,No,No,No,No,One year,No,Bank transfer (automatic),56.15,3487.95,No


## 3. Exportar crudo a `resultados/churn_crudo.csv`

In [3]:
RESULTADOS = (pathlib.Path.cwd().parent / 'resultados')
RESULTADOS.mkdir(parents=True, exist_ok=True)
ruta_crudo = RESULTADOS / 'churn_crudo.csv'
df_crudo.to_csv(ruta_crudo, index=False, encoding='utf-8')
print('Crudo exportado a:', ruta_crudo)

Crudo exportado a: G:\Mi unidad\iUDigital Software\2026 S2\ProgramacionAnalisisdeDatos\EV1\AE2\resultados\churn_crudo.csv


## 4. Exploracion descriptiva

In [4]:
print('filas y columnas:', df_crudo.shape)
print('\ncolumnas y tipos:')
print(df_crudo.dtypes)
print('\nnulos por columna:')
print(df_crudo.isna().sum())
print('\nduplicados:', df_crudo.duplicated().sum())

filas y columnas: (7043, 21)

columnas y tipos:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

nulos por columna:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0


In [5]:
df_crudo.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [6]:
for col in ['Churn', 'Contract', 'InternetService', 'PaymentMethod']:
    if col in df_crudo.columns:
        print(f'--- {col} ---')
        print(df_crudo[col].value_counts(dropna=False))
        print()

--- Churn ---
Churn
No     5174
Yes    1869
Name: count, dtype: int64

--- Contract ---
Contract
Month-to-month    3875
Two year          1695
One year          1473
Name: count, dtype: int64

--- InternetService ---
InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64

--- PaymentMethod ---
PaymentMethod
Electronic check             2365
Mailed check                 1612
Bank transfer (automatic)    1544
Credit card (automatic)      1522
Name: count, dtype: int64



## 5. Que representa cada fila y la necesidad analitica

Cada **fila es un cliente** de la compania de telecomunicaciones. Las columnas describen:

- **Perfil del cliente:** `gender`, `SeniorCitizen`, `Partner`, `Dependents`.
- **Antiguedad y contrato:** `tenure` (meses), `Contract`, `PaperlessBilling`, `PaymentMethod`.
- **Servicios contratados:** `PhoneService`, `MultipleLines`, `InternetService`, `OnlineSecurity`, `TechSupport`, `StreamingTV`, etc.
- **Facturacion:** `MonthlyCharges`, `TotalCharges`.
- **Variable objetivo:** `Churn` (si el cliente abandono el servicio).

**Necesidad analitica:** identificar que caracteristicas se asocian con el abandono (`Churn`) para anticiparlo y disenar acciones de retencion. Variables como `tenure`, `Contract`, `MonthlyCharges` y el tipo de servicio de internet respaldan directamente esa necesidad.

## 6. Transformaciones de preparacion

### T1 - Renombrar columnas a snake_case

In [7]:
import re

def a_snake_case(nombre: str) -> str:
    s = re.sub(r'(.)([A-Z][a-z]+)', r'\1_\2', str(nombre))
    s = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', s)
    return s.lower().strip().replace(' ', '_')

df_t1 = df_crudo.rename(columns={c: a_snake_case(c) for c in df_crudo.columns})
print('Columnas tras T1:', list(df_t1.columns))
df_t1.head()

Columnas tras T1: ['customer_id', 'gender', 'senior_citizen', 'partner', 'dependents', 'tenure', 'phone_service', 'multiple_lines', 'internet_service', 'online_security', 'online_backup', 'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies', 'contract', 'paperless_billing', 'payment_method', 'monthly_charges', 'total_charges', 'churn']


,customer_id,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,...,device_protection,tech_support,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### T2 - Convertir `total_charges` a numerico

`total_charges` viene como texto con cadenas vacias; se convierte con `pd.to_numeric(..., errors='coerce')`.

In [8]:
import pandas as pd

df_t2 = df_t1.copy()
df_t2['total_charges'] = pd.to_numeric(df_t2['total_charges'], errors='coerce')
nulos_total_charges = int(df_t2['total_charges'].isna().sum())
print('Nulos resultantes en total_charges:', nulos_total_charges)
df_t2[['tenure', 'monthly_charges', 'total_charges']].head()

Nulos resultantes en total_charges: 11


,tenure,monthly_charges,total_charges
0,1,29.85,29.85
1,34,56.95,1889.50
2,2,53.85,108.15
3,45,42.30,1840.75
4,2,70.70,151.65


### T3 - Tratar los nulos de `total_charges`

**Criterio:** los nulos corresponden a clientes con `tenure = 0` (sin facturacion acumulada aun). Se imputan con `0.0`, que es el valor coherente para un cliente nuevo.

In [9]:
df_t3 = df_t2.copy()
print('tenure de los registros con total_charges nulo:')
print(df_t3.loc[df_t3['total_charges'].isna(), 'tenure'].value_counts(dropna=False))
df_t3['total_charges'] = df_t3['total_charges'].fillna(0.0)
print('\nNulos en total_charges tras imputar:', int(df_t3['total_charges'].isna().sum()))
df_t3.head()

tenure de los registros con total_charges nulo:
tenure
0    11
Name: count, dtype: int64

Nulos en total_charges tras imputar: 0


,customer_id,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,...,device_protection,tech_support,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### T4 - Nueva columna `cargo_promedio_mensual`

`cargo_promedio_mensual = total_charges / tenure`, protegiendo la division por cero.

In [10]:
import numpy as np

df_t4 = df_t3.copy()
df_t4['cargo_promedio_mensual'] = np.where(
    df_t4['tenure'] > 0,
    df_t4['total_charges'] / df_t4['tenure'].replace(0, np.nan),
    0.0,
)
df_t4['cargo_promedio_mensual'] = df_t4['cargo_promedio_mensual'].fillna(0.0).round(2)
df_t4[['tenure', 'total_charges', 'cargo_promedio_mensual']].head()

,tenure,total_charges,cargo_promedio_mensual
0,1,29.85,29.85
1,34,1889.50,55.57
2,2,108.15,54.08
3,45,1840.75,40.91
4,2,151.65,75.82


### T5 - Segmentacion de antiguedad con `pd.cut`

In [11]:
df_t5 = df_t4.copy()
df_t5['segmento_antiguedad'] = pd.cut(
    df_t5['tenure'],
    bins=[-1, 12, 24, 48, float('inf')],
    labels=['0-12', '13-24', '25-48', '49+'],
)
print(df_t5['segmento_antiguedad'].value_counts())
df_transformado = df_t5
df_transformado.head()

segmento_antiguedad
49+      2239
0-12     2186
25-48    1594
13-24    1024
Name: count, dtype: int64


,customer_id,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,...,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn,cargo_promedio_mensual,segmento_antiguedad
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,29.85,0-12
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,No,One year,No,Mailed check,56.95,1889.50,No,55.57,25-48
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,54.08,0-12
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,40.91,25-48
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,75.82,0-12


## 7. Exportar transformado a `resultados/churn_transformado.csv`

In [12]:
ruta_transformado = RESULTADOS / 'churn_transformado.csv'
df_transformado.to_csv(ruta_transformado, index=False, encoding='utf-8')
print('Transformado exportado a:', ruta_transformado)
print('shape final:', df_transformado.shape)

Transformado exportado a:

 G:\Mi unidad\iUDigital Software\2026 S2\ProgramacionAnalisisdeDatos\EV1\AE2\resultados\churn_transformado.csv
shape final: (7043, 23)


## 8. Resumen automatico en `resultados/resumen_dataframe.txt`

In [13]:
summarizer = DatasetSummarizer(df_transformado, metadata=loader.get_metadata())
ruta_resumen = summarizer.save(RESULTADOS / 'resumen_dataframe.txt')
print('Resumen guardado en:', ruta_resumen)

Resumen guardado en:

 G:\Mi unidad\iUDigital Software\2026 S2\ProgramacionAnalisisdeDatos\EV1\AE2\resultados\resumen_dataframe.txt


## 9. Verificacion: leer e imprimir el `.txt` generado

In [14]:
print(ruta_resumen.read_text(encoding='utf-8'))

RESUMEN DEL DATASET
1. IDENTIFICACION
------------------------------------------------------------
   dataset_id      : scikit-learn/churn-prediction
   split           : train
   configuracion   : None
   revision        : None
   fecha_consulta  : 2026-10-24T16:02:41.145849+00:00
   es_muestra      : False

2. DIMENSIONES
------------------------------------------------------------
   filas    : 7043
   columnas : 23

3. COLUMNAS Y TIPOS
------------------------------------------------------------
   customer_id                    object
   gender                         object
   senior_citizen                 int64
   partner                        object
   dependents                     object
   tenure                         int64
   phone_service                  object
   multiple_lines                 object
   internet_service               object
   online_security                object
   online_backup                  object
   device_protection              object
   te